Path Setup

In [1]:
import sys
sys.path.append("..")

Import

In [2]:
from src.preprocessing.preprocessor import TextPreprocessor
from src.extraction.extractor import extract_text

Initialize

In [3]:
preprocessor = TextPreprocessor()
print("Preprocessor loaded successfully")
print("Stopword count:", len(preprocessor.stopwords))

2026-08-02 23:45:43 | INFO     | src.preprocessing.preprocessor | Loading spaCy model: en_core_web_sm


Preprocessor loaded successfully
Stopword count: 331


Test a hardcoded strings

In [4]:
sample_text = "The mitochondria is the powerhouse of the cell. H2O and CO2 are important molecules in Biology."
tokens = preprocessor.preprocess(sample_text)
print("Tokens:", tokens)

Tokens: ['mitochondria', 'powerhouse', 'cell', 'h2o', 'co2', 'important', 'molecule', 'biology']


Test preprocess_to_string()

In [5]:
cleaned_string = preprocessor.preprocess_to_string(sample_text)
print("Cleaned string:", cleaned_string)

Cleaned string: mitochondria powerhouse cell h2o co2 important molecule biology


Test empty/edge-case input

In [6]:
empty_result = preprocessor.preprocess("")
whitespace_result = preprocessor.preprocess("     ")
print("Empty input result:", empty_result)
print("Whitespace input result:", whitespace_result)

2026-08-02 23:45:51 | WARNING  | src.preprocessing.preprocessor | Empty text passed to preprocess()
2026-08-02 23:45:51 | WARNING  | src.preprocessing.preprocessor | Empty text passed to preprocess()


Empty input result: []
Whitespace input result: []


Test n-gram generation

In [7]:
bigrams = preprocessor.generate_ngrams(tokens, n=2)
print("Bigrams:", bigrams)

Bigrams: ['mitochondria_powerhouse', 'powerhouse_cell', 'cell_h2o', 'h2o_co2', 'co2_important', 'important_molecule', 'molecule_biology']


End-to-end test: real extracted text from Phase 2 → preprocessing

In [8]:
extraction_result = extract_text("../data/samples/ANS_PDF.pdf") # swap file names for test.
raw_text = extraction_result["text"]

real_tokens = preprocessor.preprocess(raw_text)
print("Original char count:", len(raw_text))
print("Token count after preprocessing:", len(real_tokens))
print("\nFirst 30 tokens:", real_tokens[:30])

2026-08-02 23:45:52 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples/ANS_PDF.pdf


Original char count: 752
Token count after preprocessing: 67

First 30 tokens: ['cell', 'membrane', 'structure', 'function', 'student', 'variant', 'pdf', 'submission', 'plasma', 'membrane', 'call', 'cell', 'membrane', 'flexible', 'boundary', 'enclose', 'living', 'cell', 'mainly', 'consist', 'phospholipid', 'bilayer', 'embed', 'protein', 'cholesterol', 'carbohydrate', 'molecule', 'main', 'job', 'control']


Test save_preprocessed()

In [9]:
preprocessor.save_preprocessed(real_tokens, "../data/samples/ANS_PDF.pdf")

Loop test across all sample files

In [10]:
import os

sample_dir = "../data/samples"
for fname in os.listdir(sample_dir):
    path = os.path.join(sample_dir, fname)
    result = extract_text(path)
    if "error" in result["metadata"]:
        print(f"{fname:30s} -> extraction ERROR, skipping")
        continue
    tokens = preprocessor.preprocess(result["text"])
    print(f"{fname:30s} -> {len(tokens):4d} tokens")

2026-08-02 23:45:55 | INFO     | src.extraction.docx_extractor | Attempting to extract text from DOCX: ../data/samples\ANS_DOCX.docx
2026-08-02 23:45:56 | INFO     | src.extraction.pdf_extractor | Attempting to extract text from PDF: ../data/samples\ANS_PDF.pdf


ANS_DOCX.docx                  ->   75 tokens


2026-08-02 23:45:58 | INFO     | src.extraction.ocr_extractor | Extracting text via OCR from: ../data/samples\ANS_PNG.png
2026-08-02 23:45:58 | INFO     | src.extraction.ocr_extractor | Preprocessed image: ../data/samples\ANS_PNG.png


ANS_PDF.pdf                    ->   67 tokens


2026-08-02 23:46:14 | INFO     | src.extraction.txt_extractor | Attempting to extract text from TXT: ../data/samples\ANS_TXT.txt


ANS_PNG.png                    ->   88 tokens
ANS_TXT.txt                    ->   59 tokens
